# Flow Matching: How Diffusion Works

This notebook explains the **flow matching** diffusion paradigm used by Wan 2.1 (and FLUX, Stable Diffusion 3, and other modern generators).  We will cover the forward noising process, the training objective, and the Euler-step denoising process used at inference time.

## What is Diffusion?

Diffusion models learn to generate data by learning to **reverse a noising process**.  Starting from clean data $x_0$, we progressively add noise until the signal is completely destroyed (pure Gaussian noise).  A neural network then learns to undo this corruption step by step.

### DDPM vs Flow Matching

| Aspect | DDPM (Denoising Diffusion) | Flow Matching |
|--------|---------------------------|---------------|
| Forward process | $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon$ | $x_t = (1-t)\, x_0 + t\, \epsilon$ |
| Training target | Noise $\epsilon$ | Velocity $v = \epsilon - x_0$ |
| Noise schedule | Learned $\beta$ schedule (complex) | Linear interpolation + shift (simple) |
| Denoising step | Complex variance-preserving formula | Simple Euler step |
| Convergence | Slower (1000s of steps typical) | Faster (20-50 steps sufficient) |

### Flow Matching: The Key Equations

**Forward process** (adding noise):

$$x_t = (1 - \sigma_t)\, x_0 + \sigma_t\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

where $\sigma_t \in [0, 1]$ is the noise level at time $t$.  This is just a **linear interpolation** between clean data and pure noise:
- At $\sigma = 0$: $x_t = x_0$ (completely clean)
- At $\sigma = 1$: $x_t = \epsilon$ (pure noise)

**Training target** (what the model predicts):

$$v = \epsilon - x_0$$

This is the **velocity** -- it points from the clean data toward the noise.  The model learns to predict this velocity at any noise level $\sigma_t$.

**Denoising step** (Euler method):

$$x_{t-1} = x_t + v_{\text{pred}} \cdot (\sigma_{t-1} - \sigma_t)$$

Since $\sigma$ decreases over the denoising schedule, $(\sigma_{t-1} - \sigma_t) < 0$, so we are effectively *subtracting* the predicted velocity, moving from noise toward clean data.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import torch
import numpy as np
import matplotlib.pyplot as plt

from nano_video_gen.diffusion.flow_match import FlowMatchScheduler

## Sigma Schedule and the Shift Parameter

The `shift` parameter controls how the sigma schedule is distributed across denoising steps.  The transformation is:

$$\sigma' = \frac{\text{shift} \cdot \sigma}{1 + (\text{shift} - 1) \cdot \sigma}$$

- **shift = 1**: Linear schedule (uniform spacing in sigma)
- **shift > 1**: More steps concentrated at higher noise levels
- **Wan uses shift = 5**: This biases compute toward the harder, high-noise denoising steps where the model must determine global structure

In [ ]:
# ---- Compare sigma schedules for different shift values ----
shifts = [1, 3, 5]
num_steps = 50

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#2196F3', '#FF9800', '#E91E63']

for shift, color in zip(shifts, colors):
    scheduler = FlowMatchScheduler()
    scheduler.set_timesteps(num_inference_steps=num_steps, shift=shift)

    sigmas = scheduler.sigmas.numpy()
    timesteps = scheduler.timesteps.numpy()
    step_indices = np.arange(len(sigmas))

    axes[0].plot(step_indices, sigmas, '-o', color=color, markersize=3,
                 label=f'shift={shift}', linewidth=2)
    axes[1].plot(step_indices, timesteps, '-o', color=color, markersize=3,
                 label=f'shift={shift}', linewidth=2)

axes[0].set_title('Sigma (noise level) vs Step Index', fontsize=13)
axes[0].set_xlabel('Denoising step')
axes[0].set_ylabel('Sigma ($\\sigma$)')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-0.02, 1.05)

axes[1].set_title('Timestep vs Step Index', fontsize=13)
axes[1].set_xlabel('Denoising step')
axes[1].set_ylabel('Timestep (0-1000)')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

fig.suptitle('Effect of Shift on the Noise Schedule', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("With shift=5 (Wan default), the first ~60% of denoising steps are spent")
print("at sigma > 0.5, giving the model more compute for the hardest (high-noise) steps.")
print("With shift=1, the schedule is linear and steps are evenly distributed.")

In [ ]:
# ---- Use the built-in visualization for the default shift=5 schedule ----
from nano_video_gen.visualization.viz import visualize_noise_schedule

scheduler = FlowMatchScheduler()
scheduler.set_timesteps(num_inference_steps=50, shift=5.0)

fig = visualize_noise_schedule(scheduler, figsize=(12, 4))
plt.show()

## The Forward Process: Visualizing Noising

The forward process linearly interpolates between clean data $x_0$ and Gaussian noise $\epsilon$:

$$x_t = (1 - t) \cdot x_0 + t \cdot \epsilon$$

Let us visualize this on a simple image to build intuition.

In [ ]:
# ---- Create a simple 2D pattern (checkerboard + gradient) ----
H, W = 64, 64
y_grid, x_grid = torch.meshgrid(torch.linspace(-1, 1, H), torch.linspace(-1, 1, W), indexing='ij')

# Checkerboard pattern
checker = ((torch.floor(x_grid * 4) + torch.floor(y_grid * 4)) % 2).float()
# Circular gradient
circle = 1.0 - (x_grid ** 2 + y_grid ** 2).sqrt().clamp(0, 1)
# Combine into a 3-channel image in [-1, 1]
x0 = torch.stack([checker, circle, 0.5 * (checker + circle)], dim=0)  # [3, H, W]
x0 = x0 * 2 - 1  # Scale to [-1, 1]

# Fixed noise for reproducibility
torch.manual_seed(42)
noise = torch.randn_like(x0)

# ---- Show noising at different t values ----
t_values = [0.0, 0.25, 0.5, 0.75, 1.0]

fig, axes = plt.subplots(1, len(t_values), figsize=(16, 3.5))

for ax, t in zip(axes, t_values):
    # Flow matching forward process: x_t = (1 - t) * x_0 + t * noise
    x_t = (1 - t) * x0 + t * noise

    # Convert to displayable image [0, 1]
    img = (x_t + 1) / 2  # [-1, 1] -> [0, 1]
    img = img.clamp(0, 1).permute(1, 2, 0).numpy()  # [H, W, 3]

    ax.imshow(img)
    ax.set_title(f'$t = {t:.2f}$\n$\\sigma = {t:.2f}$', fontsize=12)
    ax.axis('off')

fig.suptitle('Forward Process: $x_t = (1-t) \\cdot x_0 + t \\cdot \\epsilon$', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

print("At t=0.0: Pure clean image (x_0)")
print("At t=0.25: Mostly clean, some noise visible")
print("At t=0.50: Equal mix of signal and noise")
print("At t=0.75: Mostly noise, faint structure")
print("At t=1.00: Pure Gaussian noise")

## Training Objective

During training, we:
1. Sample a clean video $x_0$ from the dataset
2. Sample a random timestep $t \sim \text{Uniform}(0, 1)$
3. Sample noise $\epsilon \sim \mathcal{N}(0, I)$
4. Create the noisy sample: $x_t = (1 - t) \cdot x_0 + t \cdot \epsilon$
5. Ask the model to predict the velocity: $\hat{v} = f_\theta(x_t, t, \text{text})$
6. Compute the loss: $\mathcal{L} = \| \hat{v} - v \|^2$ where $v = \epsilon - x_0$

The MSE loss between the predicted and true velocity is the **only** training signal.  The model learns to predict how to transform noisy data back toward clean data at every noise level.

In [ ]:
from nano_video_gen.model.nano_dit import NanoDiT

# ---- Create a randomly initialized model and scheduler ----
torch.manual_seed(0)
model = NanoDiT()  # Default nano-scale params
model.eval()

scheduler = FlowMatchScheduler()

# ---- Compute loss on random "data" ----
# In a real setting, x0 would be a real video encoded by the VAE.
# Here we use random tensors to demonstrate the mechanics.
x0 = torch.randn(1, 4, 4, 16, 16)    # Fake latent video
context = torch.randn(1, 8, 64)        # Fake text embeddings

loss = scheduler.compute_loss(model, x0, context)

print(f"Flow matching training loss (random init): {loss.item():.4f}")
print()
print("This loss is high because the model is randomly initialized.")
print("It has no idea what velocity to predict.")
print("\nWith training, this loss would decrease as the model learns to")
print("predict the correct velocity v = noise - x_0 at each timestep.")
print(f"\nFor reference, the expected loss for a random predictor on")
print(f"standard normal data is approximately 2.0 (since Var(noise - x0) = 2).")

## The Denoising (Inference) Process

At inference time, we start from **pure noise** $x_T \sim \mathcal{N}(0, I)$ and iteratively denoise using the **Euler method**:

$$x_{t-1} = x_t + v_{\text{pred}} \cdot (\sigma_{t-1} - \sigma_t)$$

Since the sigma schedule decreases from $\sigma_T \approx 1$ to $\sigma_0 = 0$, the step size $(\sigma_{t-1} - \sigma_t)$ is **negative**.  This means we are moving in the **opposite direction** of the velocity -- from noise toward clean data.

Think of it as following a **flow field** in reverse: the velocity field $v(x_t, t)$ describes how data flows from clean to noisy, and by following it backward we recover the clean data.

### Step-by-Step Euler Integration

Given a schedule with $N$ steps and sigma values $\sigma_0 > \sigma_1 > \cdots > \sigma_{N-1} \approx 0$:

1. Start with $x \sim \mathcal{N}(0, I)$ at noise level $\sigma_0$
2. For each step $i = 0, 1, \ldots, N-1$:
   - Predict velocity: $v = f_\theta(x, \sigma_i, \text{text})$
   - Update: $x \leftarrow x + v \cdot (\sigma_{i+1} - \sigma_i)$  where $\sigma_N = 0$
3. Return $x$ as the generated sample

In [ ]:
# ---- Demonstrate a single Euler denoising step ----
torch.manual_seed(123)

# Setup the scheduler with a few steps
scheduler = FlowMatchScheduler()
scheduler.set_timesteps(num_inference_steps=10, shift=5.0)

print("Sigma schedule (10 steps, shift=5):")
print(f"  Sigmas:    {scheduler.sigmas.numpy().round(4)}")
print(f"  Timesteps: {scheduler.timesteps.numpy().round(1)}")
print()

# Current state: noisy sample at step 0 (highest noise)
x_t = torch.randn(1, 4, 4, 16, 16)  # Pure noise

# Simulate a "predicted velocity" (in practice, this comes from the DiT model)
v_pred = torch.randn_like(x_t) * 0.1  # Small random velocity for illustration

# ---- Manual Euler step calculation ----
sigma_current = scheduler.sigmas[0].item()
sigma_next = scheduler.sigmas[1].item()
delta_sigma = sigma_next - sigma_current

x_t_minus_1 = x_t + v_pred * delta_sigma

print("Single Euler denoising step:")
print(f"  sigma_t     = {sigma_current:.4f}")
print(f"  sigma_{{t-1}} = {sigma_next:.4f}")
print(f"  delta_sigma = {delta_sigma:.4f}  (negative: we are denoising)")
print()
print(f"  x_t     stats: mean={x_t.mean().item():.4f}, std={x_t.std().item():.4f}")
print(f"  v_pred  stats: mean={v_pred.mean().item():.4f}, std={v_pred.std().item():.4f}")
print(f"  x_{{t-1}} stats: mean={x_t_minus_1.mean().item():.4f}, std={x_t_minus_1.std().item():.4f}")
print()
print("Formula: x_{t-1} = x_t + v_pred * (sigma_{t-1} - sigma_t)")
print(f"         x_{{t-1}} = x_t + v_pred * {delta_sigma:.4f}")

# ---- Verify using the scheduler's step() method ----
timestep = scheduler.timesteps[0]
x_t_minus_1_check = scheduler.step(v_pred, timestep, x_t)
diff = (x_t_minus_1 - x_t_minus_1_check).abs().max().item()
print(f"\nVerification: max diff between manual and scheduler.step() = {diff:.2e}")

## Why Does Wan Use shift=5?

The sigma schedule shift controls **where the model spends its compute** during the denoising process:

- **High noise ($\sigma$ close to 1)**: The model must determine the global structure -- what objects exist, where they are, the overall composition.  This is the *hardest* part of generation.

- **Low noise ($\sigma$ close to 0)**: The model refines fine details -- textures, edges, exact colors.  This is easier because the coarse structure is already determined.

With `shift=5`, the schedule is **compressed** toward the high-noise end.  This means:
- More denoising steps are spent in the high-noise regime (where creative decisions happen)
- Fewer steps are spent in the low-noise regime (where refinement is simpler)

This is particularly important for **video** generation because:
1. Videos have more complex structure than images (temporal coherence)
2. Getting the global motion and scene layout right early is critical
3. Fine details can be resolved with fewer steps

The shift transformation is:

$$\sigma' = \frac{s \cdot \sigma}{1 + (s - 1) \cdot \sigma}$$

At the extremes: $\sigma' \to 0$ when $\sigma \to 0$ and $\sigma' \to 1$ when $\sigma \to 1$ (for any shift $s > 0$).  But in between, higher $s$ pushes the sigma values upward, allocating more steps to the high-noise regime.

## Summary

Flow matching provides a clean, elegant framework for diffusion-based generation:

| Concept | Formula | Intuition |
|---------|---------|----------|
| Forward process | $x_t = (1-\sigma)\, x_0 + \sigma\, \epsilon$ | Linear interpolation: clean $\leftrightarrow$ noise |
| Velocity target | $v = \epsilon - x_0$ | Direction from data to noise |
| Training loss | $\|\hat{v} - v\|^2$ | MSE between predicted and true velocity |
| Euler step | $x_{t-1} = x_t + \hat{v} \cdot (\sigma_{t-1} - \sigma_t)$ | Follow velocity field in reverse |
| Schedule shift | $\sigma' = \frac{s \cdot \sigma}{1 + (s-1)\sigma}$ | Allocate more compute to hard steps |

Compared to DDPM, flow matching is simpler (linear interpolation vs square-root schedules), faster to converge, and requires fewer denoising steps at inference time.